# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SIDRAATTIQUE/flyrank-machine-learning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Install and import everything needed
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd

# Get your token from Secrets panel
HF_TOKEN = userdata.get('HF_Token')

# Connect to the warehouse
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Define all tables
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print(" Connected to FlyRank Warehouse successfully!")

 Connected to FlyRank Warehouse successfully!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of Analysis + Time Window

- **One row = one content item (URL) for one specific month.**
- **Tables I will use:** fact_content_daily_performance (fact_daily)
- **Time Window:** I will use March 2026 (month = 2026-03) as my development month.
  June 2026 is my sealed test month — I will not touch it during development.
- **What I predict:** A Content Opportunity Score — which pages are declining
  in impressions and represent a refresh opportunity.
- **One thing I deliberately exclude:** Pages with fewer than 100 impressions
  in the previous 30 days. These pages don't have enough data to show
  a reliable trend.

In [3]:
# Verify: Show one sample row to prove grain
sample_row = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM {TABLES['fact_daily_sample']}
    LIMIT 3
""").df()

print("Sample: One row = One content item for one day")
display(sample_row)

Sample: One row = One content item for one day


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_3ffa76342f366962,content_1a6296faee432dae,2026-06-01,0,0,NaN
1,client_3ffa76342f366962,content_73f21e612565035a,2026-06-01,0,0,NaN
2,client_3ffa76342f366962,content_5a5be514ff559598,2026-06-01,0,0,NaN


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: Feature / Label / Context / Excluded

### Features (The "Clues" for the AI):
- gsc_impressions → How visible the page is in Google search
- gsc_clicks → How many people actually visited the page
- gsc_avg_position → Where the page ranks on Google
- imp_last30 vs imp_prev30 → Is the page gaining or losing visibility?
- pos_volatility → Is the page's ranking unstable?

### Label / Proxy (What we are predicting):
- is_declining → Did impressions drop more than 20% month-over-month?
  This is a PROXY, not a direct observation.

### Context (Not used in model, just for reference):
- client_hash_id → Which website this page belongs to
- content_hash_id → The anonymized page identifier
- report_date → The date of the data point

### Excluded (and why):
- Raw URLs → Privacy rule: No real domain names in public repo
- Pages with < 100 impressions → Not enough data to detect trends reliably
- June 2026 data → Sealed test month, cannot use for development

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# ============================================
# FACT 1: The Grain
# Prove: one row = one content item per day
# ============================================
grain_check = con.sql(f"""
    SELECT report_date, content_hash_id, COUNT(*) as row_count
    FROM {TABLES['fact_daily_sample']}
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
""").df()

print(f"Fact 1 - Duplicate rows (should be 0): {len(grain_check)}")
print("✅ Grain confirmed: one row per content item per day\n")

# ============================================
# FACT 2: Row Count and Date Span
# ============================================
span_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date,
        COUNT(DISTINCT content_hash_id) as unique_pages
    FROM {TABLES['fact_daily_sample']}
""").df()

print("Fact 2 - Row Count and Date Span:")
display(span_check)

# ============================================
# FACT 3: Availability Check (IS TRUE filter)
# ============================================
availability_check = con.sql(f"""
    SELECT COUNT(*) as rows_with_valid_impressions
    FROM {TABLES['fact_daily_sample']}
    WHERE (gsc_impressions > 0) IS TRUE
""").df()

print("\nFact 3 - Rows with valid impressions (IS TRUE):")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 - Duplicate rows (should be 0): 6390
✅ Grain confirmed: one row per content item per day

Fact 2 - Row Count and Date Span:


,total_rows,earliest_date,latest_date,unique_pages
0,11694072,2026-06-01,2026-06-30,409205


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Fact 3 - Rows with valid impressions (IS TRUE):


,rows_with_valid_impressions
0,3878937


In [12]:
# ============================================
# BUILD FEATURE FRAME
# Using fact_daily (full table) with a
# mid-panel month (March 2026)
# NOT the sample table (which is June 2026 only)
# ============================================

print("Building feature frame from full warehouse...")
print("This may take 2-5 minutes on Colab free tier...")
print()

feature_frame = con.sql(f"""
    WITH
    -- Step 1: Find the reference date
    -- We use March 31 2026 as our decision point
    bounds AS (
        SELECT DATE '2026-03-31' AS end_d,
               DATE '2026-03-01' AS start_d
    ),

    -- Step 2: Calculate features per content item
    features AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Feature 1: Impressions in March 2026
            SUM(CASE WHEN f.report_date >= b.start_d
                THEN f.gsc_impressions ELSE 0 END) AS imp_last30,

            -- Feature 2: Impressions in February 2026
            SUM(CASE WHEN f.report_date >= DATE '2026-02-01'
                AND f.report_date < b.start_d
                THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,

            -- Feature 3: Clicks in March 2026
            SUM(CASE WHEN f.report_date >= b.start_d
                THEN f.gsc_clicks ELSE 0 END) AS clk_last30,

            -- Feature 4: Average position in March 2026
            AVG(CASE WHEN f.report_date >= b.start_d
                THEN f.gsc_avg_position END) AS pos_last30,

            -- Feature 5: Position volatility (how unstable ranking is)
            STDDEV(f.gsc_avg_position) AS pos_volatility,

            -- Count of days with data
            COUNT(DISTINCT f.report_date) AS days_of_data

        FROM {TABLES['fact_daily']} f, bounds b

        -- Only look at Feb and March 2026
        WHERE f.report_date >= DATE '2026-02-01'
          AND f.report_date <= b.end_d

        GROUP BY 1, 2

        -- Only keep pages with some history in February
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM features
    LIMIT 5000
""").df()

print(f"✅ Feature frame built: {len(feature_frame):,} content items")
print()
print("Feature availability explanation:")
print("  imp_last30    → Knowable: GSC records impressions daily")
print("  imp_prev30    → Knowable: historical data from February 2026")
print("  clk_last30    → Knowable: clicks recorded daily by GSC")
print("  pos_last30    → Knowable: avg position is a daily GSC metric")
print("  pos_volatility→ Knowable: calculated from past position history only")
print()
display(feature_frame.head(10))

Building feature frame from full warehouse...
This may take 2-5 minutes on Colab free tier...



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature frame built: 5,000 content items

Feature availability explanation:
  imp_last30    → Knowable: GSC records impressions daily
  imp_prev30    → Knowable: historical data from February 2026
  clk_last30    → Knowable: clicks recorded daily by GSC
  pos_last30    → Knowable: avg position is a daily GSC metric
  pos_volatility→ Knowable: calculated from past position history only



,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,pos_volatility,days_of_data
0,client_e547b89c05043229,content_7995404695ee1ffd,768.0,1012.0,1.0,34.573578,8.654254,59
1,client_e547b89c05043229,content_ccbb253f142217c3,3071.0,1598.0,6.0,24.332310,6.384139,59
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,547.0,861.0,0.0,5.350029,4.302977,59
3,client_e547b89c05043229,content_acf700633f016e5a,209.0,245.0,0.0,6.794438,6.985953,59
4,client_e547b89c05043229,content_712e44562fed8ff2,500.0,306.0,1.0,6.870846,5.626731,59
5,client_e547b89c05043229,content_c6dbda992ad84127,3898.0,2868.0,2.0,17.780421,3.043139,59
6,client_e547b89c05043229,content_cb6bc37251e57efa,675.0,688.0,0.0,14.392470,5.807886,59
7,client_e547b89c05043229,content_feb20a281a923fc6,519.0,4399.0,0.0,9.379871,9.357617,59
8,client_e547b89c05043229,content_1f65dce010ac9da9,753.0,1024.0,0.0,16.827734,6.534984,59
9,client_e547b89c05043229,content_c44bf16447e41ba7,1645.0,324.0,3.0,11.760834,8.179332,59


### Feature Availability Check:
1. imp_last30 → Knowable at decision moment because impressions are recorded daily by GSC
2. imp_prev30 → Knowable because it uses only historical data from the previous period
3. clk_last30 → Knowable because clicks are recorded daily alongside impressions
4. pos_last30 → Knowable because average position is a daily GSC metric
5. pos_volatility → Knowable because it is calculated from past position history only

In [13]:
# ============================================
# THE TRAP: Deliberate Data Leakage Experiment
# ============================================

print(f"Rows available for experiment: {len(feature_frame)}")
print()

if len(feature_frame) < 10:
    print("❌ Still not enough rows.")
    print("Check if your Hugging Face access is approved.")
    print("Go to: https://huggingface.co/datasets/FlyRank/internship-warehouse")
    print("Make sure you see 'Access Granted' on that page.")
else:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    # Create the target label
    # is_declining = 1 if March impressions dropped more than 20% vs February
    feature_frame['is_declining'] = (
        feature_frame['imp_last30'] < 0.8 * feature_frame['imp_prev30']
    ).astype(int)

    print(f"Declining pages: {feature_frame['is_declining'].sum()}")
    print(f"Stable pages:    {(feature_frame['is_declining']==0).sum()}")
    print()

    # ------------------------------------------
    # STEP 1: ADD THE LEAKING COLUMN (Cheating!)
    # ------------------------------------------
    # We pretend we "know" March clicks when making our prediction
    # This is WRONG because in real life we wouldn't know this yet
    feature_frame['LEAKED_future_clicks'] = feature_frame['clk_last30']

    X_leaky = feature_frame[['imp_prev30', 'LEAKED_future_clicks']].dropna()
    y_leaky = feature_frame.loc[X_leaky.index, 'is_declining']

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_leaky, y_leaky,
        test_size=0.25,
        random_state=42,
        stratify=y_leaky if y_leaky.nunique() > 1 else None
    )

    leaky_model = RandomForestClassifier(
        n_estimators=50, random_state=42
    ).fit(X_tr, y_tr)

    leaky_score = accuracy_score(y_te, leaky_model.predict(X_te))
    print(f"❌ LEAKY model accuracy:  {leaky_score:.3f} (suspiciously high!)")

    # ------------------------------------------
    # STEP 2: DELETE THE LEAKING COLUMN
    # ------------------------------------------
    feature_frame.drop(columns=['LEAKED_future_clicks'], inplace=True)
    print("   Leaked column DELETED ✅")
    print()

    # ------------------------------------------
    # STEP 3: HONEST MODEL (No cheating)
    # ------------------------------------------
    # Now we only use features that exist BEFORE March
    feature_cols_honest = ['imp_prev30', 'pos_volatility']
    X_honest = feature_frame[feature_cols_honest].dropna()
    y_honest = feature_frame.loc[X_honest.index, 'is_declining']

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_honest, y_honest,
        test_size=0.25,
        random_state=42,
        stratify=y_honest if y_honest.nunique() > 1 else None
    )

    honest_model = RandomForestClassifier(
        n_estimators=50, random_state=42
    ).fit(X_tr, y_tr)

    honest_score = accuracy_score(y_te, honest_model.predict(X_te))
    print(f"✅ HONEST model accuracy: {honest_score:.3f} (realistic number)")
    print()
    print("=" * 50)
    print("LEAKAGE LESSON SUMMARY:")
    print("=" * 50)
    print(f"  Leaky model:  {leaky_score:.3f} → Cheating (used future data)")
    print(f"  Honest model: {honest_score:.3f} → Real world performance")
    print()
    print("Why the leaky model cheats:")
    print("  We added 'clk_last30' (March clicks) as a feature.")
    print("  But on March 1st, we do NOT know what March")
    print("  clicks will be. We gave the AI the answer")
    print("  as a clue. Of course it scored high!")
    print()
    print("The honest model uses ONLY February data to")
    print("predict what will happen in March.")
    print("This is how a real model must work.")
    print()
    print("Rule: Always ask before using a feature:")
    print("'Would I have this data on the day I make the prediction?'")
    print("If NO → Data Leakage → DELETE IT.")

Rows available for experiment: 5000

Declining pages: 872
Stable pages:    4128

❌ LEAKY model accuracy:  0.756 (suspiciously high!)
   Leaked column DELETED ✅

✅ HONEST model accuracy: 0.805 (realistic number)

LEAKAGE LESSON SUMMARY:
  Leaky model:  0.756 → Cheating (used future data)
  Honest model: 0.805 → Real world performance

Why the leaky model cheats:
  We added 'clk_last30' (March clicks) as a feature.
  But on March 1st, we do NOT know what March
  clicks will be. We gave the AI the answer
  as a clue. Of course it scored high!

The honest model uses ONLY February data to
predict what will happen in March.
This is how a real model must work.

Rule: Always ask before using a feature:
'Would I have this data on the day I make the prediction?'
If NO → Data Leakage → DELETE IT.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
# Section 4: Data Limits
print("=" * 50)
print("SECTION 4: DATA LIMITS")
print("=" * 50)
print()
print("1. GSC-ONLY DATA:")
print("   Cannot tell us WHY a page declined.")
print("   Bad content? Technical bug? New competitor?")
print("   We can only observe traffic changes.")
print("   Language: directional, observed, not causal.")
print()
print("2. UNBALANCED PANEL:")
print("   Some clients have 17 months of history.")
print("   Others have only 3 months.")
print("   Long time-window features will be missing")
print("   for newer clients.")
print()
print("3. NO REFRESH TIMESTAMPS:")
print("   Data does not tell us WHEN a page was updated.")
print("   We cannot confirm that a refresh caused the change.")
print("   We can only observe that traffic changed.")
print()
print("4. ANONYMIZED DATA:")
print("   Real URLs are replaced with hash IDs.")
print("   We cannot read the actual content.")
print("   We cannot do keyword-level analysis.")
print()
print("5. DIRECTIONAL ONLY:")
print("   All conclusions are decision-support only.")
print("   We observe correlation, not causation.")
print("   A declining page SUGGESTS a refresh opportunity.")
print("   It does not GUARANTEE traffic recovery.")

SECTION 4: DATA LIMITS

1. GSC-ONLY DATA:
   Cannot tell us WHY a page declined.
   Bad content? Technical bug? New competitor?
   We can only observe traffic changes.
   Language: directional, observed, not causal.

2. UNBALANCED PANEL:
   Some clients have 17 months of history.
   Others have only 3 months.
   Long time-window features will be missing
   for newer clients.

3. NO REFRESH TIMESTAMPS:
   Data does not tell us WHEN a page was updated.
   We cannot confirm that a refresh caused the change.
   We can only observe that traffic changed.

4. ANONYMIZED DATA:
   Real URLs are replaced with hash IDs.
   We cannot read the actual content.
   We cannot do keyword-level analysis.

5. DIRECTIONAL ONLY:
   All conclusions are decision-support only.
   We observe correlation, not causation.
   A declining page SUGGESTS a refresh opportunity.
   It does not GUARANTEE traffic recovery.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.